In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/train.csv")

X = df.drop(columns=['target', 'ID_code'])
y = df['target']

# Recreate engineered features
X['mean'] = X.mean(axis=1)
X['std'] = X.std(axis=1)
X['min'] = X.min(axis=1)
X['max'] = X.max(axis=1)
X['range'] = X['max'] - X['min']

C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_27580\1445989574.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['mean'] = X.mean(axis=1)
C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_27580\1445989574.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['std'] = X.std(axis=1)
C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_27580\1445989574.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joi

In [2]:
from sklearn.ensemble import RandomForestClassifier

# Train model for feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Get importance
importances = rf.feature_importances_

# Create dataframe
feat_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
})

# Sort features
feat_importance = feat_importance.sort_values(by='importance', ascending=False)

feat_importance.head(10)

,feature,importance
81,var_81,0.012104
139,var_139,0.009518
12,var_12,0.009482
53,var_53,0.008523
110,var_110,0.008448
26,var_26,0.008406
174,var_174,0.008257
146,var_146,0.008246
166,var_166,0.007800
22,var_22,0.007631


In [ ]:
# Select top 50 features
top_features = feat_importance['feature'].head(100).tolist()

X_selected = X[top_features]

X_selected.shape

(200000, 50)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression(max_iter=1000, class_weight='balanced')

model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.6).astype(int)

print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

F1: 0.39108034132989855
ROC-AUC: 0.8072523154101643


d:\6th semester study\Regrerssion-Comparison\research\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 🧪 Feature Selection Experiments

### Objective
To reduce dimensionality by selecting the most important features using model-based importance.

---

### Approach

- Used Random Forest to compute feature importance
- Selected top 50 most important features
- Trained Logistic Regression on reduced feature set

---

### Results

| Stage | F1-score | ROC-AUC |
|------|----------|--------|
| Feature Engineering | ~0.47 | ~0.864 |
| Feature Selection (Top 50) | ~0.39 | ~0.807 |

---

### Key Observations

- Performance decreased after feature selection
- Both F1-score and ROC-AUC dropped significantly
- Important information was lost during feature reduction

---

### Insight

- Logistic Regression benefits from high-dimensional feature space
- Some low-importance features still contribute useful signals
- Feature importance from tree-based models may not align with linear models

---

### Conclusion

- Feature selection is not beneficial for Logistic Regression in this case
- Full feature set with engineered features performs better
- Feature selection will be more effective when used with tree-based models